## ANN example:

In [19]:
#手机功能（共20个） -> 售价（范围0, 1, 2, 3）

import torch
from torch import nn, optim
from torch.utils.data import TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from torch.utils.data import TensorDataset, DataLoader
import time
from torchsummary import summary



def create_dataset():
    data = pd.read_csv('D:\code\python\deep learning\day5\data\price_data.csv')
    x, y = data.iloc[:, :-1], data.iloc[:, -1]
    x = x.astype(np.float32)

    x_train, x_test, y_train, y_test = train_test_split(x, y,
                                                         test_size=0.2,
                                                         random_state=23,
                                                         stratify=y)
    transfer = StandardScaler()
    x_train = transfer.fit_transform(x_train)
    x_test = transfer.transform(x_test)
    train_dataset = TensorDataset(torch.tensor(x_train), torch.tensor(y_train.values))
    test_dataset = TensorDataset(torch.tensor(x_test), torch.tensor(y_test.values))
    return train_dataset, test_dataset, x_train.shape[1], len(np.unique(y))

class phone_price_model(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        #hidden layer 1: 20 -> 128
        self.linear1 = nn.Linear(input_dim, 128)
        self.dropout1 = nn.Dropout(p=0.5)
        #hidden layer 2: 128 -> 256
        self.linear2 = nn.Linear(128, 256)
        self.dropout2 = nn.Dropout(p=0.5)
        #hidden layer 3: 256 -> 128
        self.linear3 = nn.Linear(256, 128)
        self.dropout3 = nn.Dropout(p=0.5)
        #output layer 3: 128 -> 4
        self.output = nn.Linear(128, output_dim)
        pass

    def forward(self, x):
        x = torch.relu(self.linear1(x))
        x = self.dropout1(x)
        x = torch.relu(self.linear2(x))
        x = self.dropout2(x)
        x = torch.relu(self.linear3(x))
        x = self.dropout3(x)
        #apply cross entropy loss
        x = self.output(x)
        return x

def training(train, input_dim, output_dim):
    train_loader = DataLoader(train, batch_size=16, shuffle=True)
    model = phone_price_model(input_dim, output_dim)
    #read existed weight data during training
    #model.load_state_dict(torch.load('D:\code\python\deep learning\day5\model\phone.pth'))
    model.train()
    loss_function = nn.CrossEntropyLoss()
    optimizer = optim.Adam(params=model.parameters(), lr=0.0001)
    epochs = 100
    #training
    for i in range(epochs):
        total_loss, batch_num = 0.0, 0
        start = time.time()
        for x, y in train_loader:
            z = model(x)
            loss = loss_function(z, y)
            optimizer.zero_grad()
            loss.sum().backward()
            optimizer.step()
            total_loss += loss.item()
            batch_num += 1
        print(f'epoch = {i}, ')
        print(f'time = {time.time() - start:.2f}, ')
        print(f'loss = {total_loss / batch_num}\n')
    # model.state_dict(): parameters of model
    torch.save(model.state_dict(), 
               'D:\code\python\deep learning\day5\model\phone.pth')
    pass



def testing(test, input_dim, output_dim):
    model = phone_price_model(input_dim, output_dim)
    model.load_state_dict(torch.load('D:\code\python\deep learning\day5\model\phone.pth'))
    test_loader = DataLoader(test, batch_size=8, shuffle=False)
    correct = 0
    for x, y in test_loader:
        model.eval()
        z = model(x)
        z = torch.argmax(z, dim=1)
        correct += (z == y).sum()
        print(f'{(z == y).sum()}\n')
    print(f'acc = {correct / len(test):.4f}')
    pass



if __name__ == '__main__':
    train, test, input_dim, output_dim = create_dataset()
    model = phone_price_model(input_dim, output_dim)
    summary(model, input_size=(16, input_dim))
    #train = model.forward(train)
    training(train, input_dim, output_dim)
    testing(test, input_dim, output_dim)

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Linear-1              [-1, 16, 128]           2,688
           Dropout-2              [-1, 16, 128]               0
            Linear-3              [-1, 16, 256]          33,024
           Dropout-4              [-1, 16, 256]               0
            Linear-5              [-1, 16, 128]          32,896
           Dropout-6              [-1, 16, 128]               0
            Linear-7                [-1, 16, 4]             516
Total params: 69,124
Trainable params: 69,124
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 0.13
Params size (MB): 0.26
Estimated Total Size (MB): 0.39
----------------------------------------------------------------
epoch = 0, 
time = 0.84, 
loss = 1.38403093457222

epoch = 1, 
time = 0.60, 
loss = 1.3765278434753419

epoc